# DX UMAP — paper figure (2 row × 6 col)

**한 노트북에 한 figure**: PTB-XL + ZZU 호환 12개 진단 코드를 4 모델 (ECG-CPC, ECG-FM, ECG-JEPA, MoRyECG(Ours)) × age subgroup (combined / adult / pediatric) UMAP 으로 시각화.

- layout: 2 row × 6 col (가로로 길게)
- 제목 X · metrics 표 X · legend 별도 figure
- 저장: `results/<YYYYMMDD_HHMMSS>/dx_umap.{pdf,svg,png}` + `dx_umap_legend.{...}`
- 논문용: PDF/SVG = vector (축소해도 무손실), 폰트 임베딩 (Illustrator 편집 가능)

In [ ]:
# === Setup: 이 한 셀로 아래 호출에 필요한 모든 것 준비 ===
import sys, os, datetime
from pathlib import Path
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
import matplotlib.text as mtext
import scripts.umap_view as uv
from scripts.umap_view import quick_dx, dx_compatibility_table, ICD_DISPLAY

plt.rcParams.update({
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.0,
    'figure.autolayout': False,
})

PROJECT_ROOT = Path(os.path.abspath('..'))
RESULTS_ROOT = PROJECT_ROOT / 'results'

compat = dx_compatibility_table()
compat_codes = compat[compat['compatible']]['icd'].tolist()

# 텍스트 전부 제거 — UMAP panel 만 (모든 Text artist 의 text 를 강제로 비움)
def quick_dx_pretty(*args,
                    figsize_per_cell=(1.9, 1.9),
                    dpi=110,
                    intra_in=0.05,
                    inter_in=0.20,
                    vrow_in=0.10,
                    margin_in=0.05,
                    save_name='dx_umap',
                    save_dir=None,
                    **kwargs):
    kwargs['show'] = False
    kwargs['show_metrics'] = False
    kwargs.setdefault('figsize_per_cell', figsize_per_cell)

    _ol, _os = Figure.legend, Figure.suptitle
    Figure.legend = lambda self, *a, **kw: None
    Figure.suptitle = lambda self, *a, **kw: None
    try:
        fig, metrics = quick_dx(*args, **kwargs)
    finally:
        Figure.legend, Figure.suptitle = _ol, _os

    fig.set_dpi(dpi)
    n_age = 3
    n_models = len(fig.axes) // n_age
    models_per_row = 2
    tgt_rows = (n_models + models_per_row - 1) // models_per_row
    raw_axes = list(fig.axes)

    # inch 단위 layout
    cw, ch = figsize_per_cell
    fig_w_in = (margin_in + n_age * cw + (n_age - 1) * intra_in + inter_in
                + n_age * cw + (n_age - 1) * intra_in + margin_in)
    fig_h_in = margin_in + tgt_rows * ch + (tgt_rows - 1) * vrow_in + margin_in
    fig.set_size_inches(fig_w_in, fig_h_in)

    group_w_in = n_age * cw + (n_age - 1) * intra_in
    for i, ax in enumerate(raw_axes):
        m, a = i // n_age, i % n_age
        nr = m // models_per_row
        m_in_row = m % models_per_row
        x_in = margin_in + m_in_row * (group_w_in + inter_in) + a * (cw + intra_in)
        y_in = margin_in + (tgt_rows - 1 - nr) * (ch + vrow_in)
        ax.set_position([x_in / fig_w_in, y_in / fig_h_in,
                         cw / fig_w_in, ch / fig_h_in])

    # ── 모든 텍스트 강제 제거 (positioning 끝난 뒤 마지막에) ──
    for ax in raw_axes:
        ax.set_title('')
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_xticks([]); ax.set_yticks([])
        ax.tick_params(labelbottom=False, labelleft=False,
                       labeltop=False, labelright=False)
        for sp in ax.spines.values():
            sp.set_visible(False)
        for t in list(ax.texts):
            t.remove()
    for t in list(fig.texts):
        t.remove()
    if getattr(fig, '_suptitle', None) is not None:
        try: fig._suptitle.remove()
        except Exception: pass
        fig._suptitle = None
    # nuclear: 남은 모든 Text artist 의 text 를 비움
    for artist in fig.findobj(match=mtext.Text):
        if artist.get_text():
            artist.set_text('')

    plt.show()

    # 저장
    if save_dir is None:
        save_dir = RESULTS_ROOT / datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    for ext, save_dpi in [('pdf', None), ('svg', None), ('png', 600)]:
        kw = {'dpi': save_dpi} if save_dpi else {}
        fig.savefig(save_dir / f'{save_name}.{ext}', **kw)
    print(f'[saved] {save_dir}/{save_name}.{{pdf,svg,png}}')

    return fig, metrics, save_dir

print('compat_codes:', compat_codes)
print(f'output base: {RESULTS_ROOT}/<YYYYMMDD_HHMMSS>/')

In [ ]:
# 호환 12개 진단 — 진단당 max 500 샘플로 balance, 2x6 layout, 텍스트 X
fig, metrics, save_dir = quick_dx_pretty(
    models=['CPC', 'ECG-FM', 'ECG-JEPA', 'Ours-cb1024'],
    include_codes=compat_codes,
    age_split=18.0,
    balance_per_code=500,
    save_name='dx_umap_compat12',
)